# Recursive Language Model (RLM) com DSPy

Neste notebook será apresentado o módulo **RLM (`dspy.RLM`)** do DSPy.

O objetivo é utilizar o texto integral do livro **Dom Casmurro**, de Machado de Assis, como contexto para que um modelo de linguagem consiga responder perguntas que exigem a análise de diferentes partes da obra.

Diferentemente de uma abordagem tradicional de **RAG (Retrieval-Augmented Generation)**, não criaremos previamente:

- chunks;
- embeddings;
- banco vetorial;
- mecanismo de busca semântica.

Em vez disso, o **RLM permite que o próprio modelo explore programaticamente o contexto**, escrevendo e executando código para localizar e analisar as informações necessárias para responder à pergunta.

## 1. O que é um RLM?

**RLM** significa **Recursive Language Model**.

A ideia central é permitir que um modelo de linguagem trabalhe com contextos muito grandes sem precisar enviar todo o conteúdo diretamente a cada chamada do LLM.

O contexto é disponibilizado ao modelo como uma variável dentro de um ambiente de execução.

O modelo pode então executar operações como:

- buscar palavras ou expressões;
- localizar posições dentro do texto;
- extrair trechos específicos;
- utilizar expressões regulares;
- dividir o texto em partes;
- contar ocorrências;
- comparar diferentes trechos;
- chamar outros LLMs para analisar partes menores do contexto.

De maneira simplificada, podemos representar o funcionamento como:

`Contexto → RLM → Código → Exploração do contexto → Subconsultas ao LLM → Resposta`

Portanto, diferentemente de um pipeline em que nós definimos previamente como as informações serão recuperadas, no RLM o próprio modelo pode decidir **como investigar o contexto**.

## 2. RLM versus RAG

É importante não confundir um **Recursive Language Model** com um sistema tradicional de **Retrieval-Augmented Generation (RAG)**.

Em um RAG, normalmente construímos previamente um mecanismo de recuperação:

`Documento → Chunks → Embeddings → Vector Store → Retrieval → LLM`

Quando uma pergunta é realizada, o sistema procura os chunks semanticamente mais próximos da pergunta e envia esses trechos ao modelo.

No RLM, a estratégia é diferente:

`Documento → Variável de contexto → RLM → Exploração programática → LLM`

O próprio modelo pode escrever código para decidir:

1. o que procurar;
2. onde procurar;
3. quais trechos são relevantes;
4. quais trechos precisam de uma análise mais profunda;
5. quando realizar novas chamadas ao modelo de linguagem.

Isso torna o RLM especialmente interessante para experimentos envolvendo **contextos longos e tarefas que exigem múltiplas etapas de investigação**.

In [1]:
import os
from dotenv import load_dotenv
import dspy

load_dotenv()

True

In [2]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Configura o modelo padrão utilizado pelo DSPy
dspy.configure(lm=lm)

## Carregando o contexto

Nosso contexto será o texto integral de **Dom Casmurro**, de Machado de Assis.

O arquivo `dom_casmurro.txt` contém a obra completa.

Neste experimento não realizaremos divisão manual do documento em chunks.

O conteúdo inteiro será carregado em uma variável Python chamada `dom_casmurro`.

Essa variável será posteriormente fornecida ao `dspy.RLM`, permitindo que o modelo explore o livro programaticamente.

Ref.: https://www.gutenberg.org/ebooks/55752

In [3]:
# Carrega todo o conteúdo de Dom Casmurro
with open("dom_casmurro.txt", "r", encoding="utf-8") as arquivo:
    dom_casmurro = arquivo.read()

print(f"Quantidade de caracteres: {len(dom_casmurro):,}")
print()
print(dom_casmurro[500:1500])

Quantidade de caracteres: 400,943

Dom Casmurro

Author: Machado de Assis


        
Release date: October 15, 2017 [eBook #55752]
                Most recently updated: October 23, 2024

Language: Portuguese

Other information and formats: www.gutenberg.org/ebooks/55752

Credits: Produced by Laura Natal Rodriguez & Marc D'Hooghe at Free
        Literature (online soon in an extended version,also linking
        to free sources for education worldwide ... MOOC's,
        educational materials,...) (Images generously made available
        by the Bibliotheca Nacional Digital Brasil.)


*** START OF THE PROJECT GUTENBERG EBOOK DOM CASMURRO ***

DOM CASMURRO

POR

MACHADO DE ASSIS

DA ACADEMIA BRAZILEIRA

H. GARNIER, LIVREIRO-EDITOR

RUA MOREIRA CEZAR, 71

RIO DE JANEIRO

6, RUE DES SAINTS-PÈRES, 6

PARIZ




I

Do titulo.

Uma noite destas, vindo da cidade para o Engenho Novo, encontrei no
trem da Central um rapaz aqui do bairro, que eu conheço de vista e
de chapéo. Comprimentou-me, sent

## 6. Definindo a Signature

Assim como outros módulos do DSPy, o `RLM` pode utilizar uma **Signature** para definir claramente quais informações entram e quais informações devem ser produzidas.

Neste exemplo teremos duas entradas:

- `contexto`: o texto integral de *Dom Casmurro*;
- `pergunta`: a questão que queremos investigar.

E uma saída:

- `resposta`: a resposta produzida pelo RLM após explorar o contexto.

A Signature também funciona como uma descrição da tarefa que queremos que o modelo execute.

In [4]:
class AnaliseLiteraria(dspy.Signature):
    """
    Responda à pergunta utilizando exclusivamente as informações
    encontradas no texto fornecido como contexto.

    Analise diferentes partes do texto quando necessário e apresente
    uma resposta fundamentada na obra.
    """

    # Livro completo
    contexto: str = dspy.InputField(
        desc="Texto integral da obra literária"
    )

    # Questão que queremos investigar
    pergunta: str = dspy.InputField(
        desc="Pergunta sobre a obra"
    )

    # Resposta produzida após exploração do contexto
    resposta: str = dspy.OutputField(
        desc="Resposta detalhada e fundamentada no texto"
    )

## Criando o Recursive Language Model

Agora podemos criar o módulo `dspy.RLM`.

O RLM trabalha de maneira iterativa.

Em cada iteração, o modelo pode:

1. analisar o estado atual do problema;
2. escrever código;
3. executar esse código;
4. observar o resultado;
5. decidir qual deve ser o próximo passo.

Esse processo pode continuar até que o modelo considere que possui informações suficientes para produzir a resposta final.

Alguns parâmetros importantes são:

- `max_iters`: quantidade máxima de ciclos de exploração;
- `max_llm_calls`: quantidade máxima de chamadas adicionais ao LLM;
- `max_output_chars`: limita o tamanho das saídas produzidas pelo ambiente de execução;
- `verbose`: permite acompanhar mais detalhes da execução.

Esses limites são importantes porque um RLM possui maior autonomia do que uma simples chamada de `dspy.Predict`.

In [5]:
analista = dspy.RLM(
    AnaliseLiteraria,

    # Número máximo de ciclos em que o modelo pode:
    # pensar -> escrever código -> executar -> observar resultado
    max_iters=15,

    # Quantidade máxima de chamadas recursivas ao LLM
    # através de llm_query() ou llm_query_batched()
    max_llm_calls=30,

    # Limite de caracteres retornados pelas execuções do REPL
    # para evitar que uma impressão gigantesca seja enviada ao modelo
    max_output_chars=10_000,

    # Mostra detalhes da execução do RLM.
    # É especialmente interessante enquanto estamos estudando.
    verbose=True,
)

## Definindo a pergunta

Para observar melhor o funcionamento do RLM, é interessante utilizar uma pergunta que não possa ser respondida simplesmente encontrando uma única frase no livro.

A pergunta abaixo exige a análise de acontecimentos distribuídos por diferentes partes da narrativa.

Queremos investigar a evolução do ciúme de Bentinho.

Para responder adequadamente, o modelo provavelmente precisará localizar e relacionar diferentes episódios envolvendo personagens como:

- Bentinho;
- Capitu;
- Escobar.

Essa característica torna a pergunta adequada para observar a estratégia de exploração criada pelo RLM.

In [6]:
pergunta = """
Ao longo de Dom Casmurro, como evolui o ciúme de Bentinho em relação
a Capitu?

Identifique episódios importantes em diferentes momentos da narrativa
que demonstrem essa evolução e explique como eles contribuem para a
desconfiança de Bentinho sobre Capitu.
"""

## Executando o RLM

Agora executaremos o módulo fornecendo:

- o livro completo como `contexto`;
- nossa questão como `pergunta`.

É importante perceber que não estamos informando ao modelo **como procurar a resposta**.

Não especificamos:

- palavras-chave;
- expressões regulares;
- capítulos;
- chunks;
- buscas semânticas.

O próprio RLM deverá construir uma estratégia para investigar o conteúdo de `dom_casmurro`.

In [7]:
resultado = analista(
    contexto=dom_casmurro,
    pergunta=pergunta
)

resultado

2026/09/07 16:07:48 INFO dspy.predict.rlm: RLM iteration 1/15
Reasoning: Vou explorar o texto (contexto) para localizar trechos e episódios relevantes que mostrem a evolução do ciúme de Bentinho em relação a Capitu. Primeiro vou buscar ocorrências de palavras-chave: "Capitu", "Bentinho", "Bento", "Escobar", "Ezequiel", "olhos de ressaca", "ciúme", "suspeita", "desconfiança". Em seguida extrairei trechos em volta das ocorrências mais importantes para analisar e depois organizarei os episódios em ordem cronológica explicando como cada um contribui para a desconfiança de Bentinho.

Agora vou executar código para localizar e mostrar amostras desses trechos. Vou limitar as saídas para facilitar a leitura.
Code:
```python
# Exploratory search in the provided contexto
print("Tamanho do texto:", len(contexto))

# Keywords to search
keywords = ["Capitu", "Bentinho", "Bento", "Escobar", "Ezequiel", "olhos de ressaca", "ciúme", "suspeita", "desconfiança", "Capitu e Escobar"]
results = {}

for kw 

Prediction(
    resposta='Resposta — evolução do ciúme de Bentinho por Capitu (fundamentada no texto):\n\n1) A fase inicial — atração e encanto:\n\n- Episódio da inscrição: Foi o mesmo que accender em mim o desejo de ler o que era.     XIV  A inscripção.  Tudo o que contei no fim do outro capitulo foi obra de um instante. O que se lhe seguiu foi ainda mais rapido. Dei um pulo, e antes que ella raspasse o muro, li estes dous nomes, abertos ao prego, o assim dispostos:  BENTO CAPITOLINA  Voltei-me para ella; Capitú tinha os olhos no chão. Ergueu-os logo, devagar, e ficámos a olhar um para o outro... Confissão de creanças, tu valias bem duas ou tres paginas, mas quero ser poupado. Em verdade, não falámos nada; o muro falou por nós. Não nos movemos, as mãos é que se estenderam pouco\n  > Explica-se que esse episódio (os nomes escritos no muro) simboliza o começo do laço íntimo entre eles: foi um gesto infantil que funda a sua afeição.\n\n- Descrição dos olhos: subir ao throno aos quinze an

## Analisando a resposta

O resultado retornado pelo RLM é um objeto `Prediction` do DSPy.

O campo `resposta` corresponde à saída que definimos anteriormente em nossa `Signature`.

A resposta representa a síntese final realizada pelo modelo após executar sua estratégia de exploração do contexto.

In [8]:
print("=" * 80)
print("RESPOSTA")
print("=" * 80)

print(resultado.resposta)

RESPOSTA
Resposta — evolução do ciúme de Bentinho por Capitu (fundamentada no texto):

1) A fase inicial — atração e encanto:

- Episódio da inscrição: Foi o mesmo que accender em mim o desejo de ler o que era.     XIV  A inscripção.  Tudo o que contei no fim do outro capitulo foi obra de um instante. O que se lhe seguiu foi ainda mais rapido. Dei um pulo, e antes que ella raspasse o muro, li estes dous nomes, abertos ao prego, o assim dispostos:  BENTO CAPITOLINA  Voltei-me para ella; Capitú tinha os olhos no chão. Ergueu-os logo, devagar, e ficámos a olhar um para o outro... Confissão de creanças, tu valias bem duas ou tres paginas, mas quero ser poupado. Em verdade, não falámos nada; o muro falou por nós. Não nos movemos, as mãos é que se estenderam pouco
  > Explica-se que esse episódio (os nomes escritos no muro) simboliza o começo do laço íntimo entre eles: foi um gesto infantil que funda a sua afeição.

- Descrição dos olhos: subir ao throno aos quinze annos. Tudo era materia ás

## Inspecionando a trajetória de execução

Uma das partes mais interessantes do `dspy.RLM` é a possibilidade de analisar sua **trajetória de execução**.

Enquanto um `dspy.Predict` tradicional normalmente realiza uma chamada ao modelo para produzir uma resposta, o RLM pode realizar diversas etapas intermediárias.

Essas etapas são armazenadas em `trajectory`.

A trajetória permite observar como o modelo tentou resolver o problema.

Por exemplo, podemos encontrar operações relacionadas a:

- buscas dentro do texto;
- extração de trechos;
- contagem de ocorrências;
- manipulação de strings;
- chamadas adicionais ao LLM;
- análise intermediária de evidências.

Portanto, ao estudar RLM, não devemos observar apenas a resposta final.

A trajetória é fundamental para compreender **a estratégia criada pelo próprio modelo para investigar o contexto**.

In [9]:
print(f"Quantidade de etapas: {len(resultado.trajectory)}")

Quantidade de etapas: 7


In [10]:
for i, etapa in enumerate(resultado.trajectory, start=1):

    print("\n" + "=" * 80)
    print(f"ETAPA {i}")
    print("=" * 80)

    print("\nCÓDIGO EXECUTADO:")
    print("-" * 80)
    print(etapa.get("code", ""))

    print("\nSAÍDA DO CÓDIGO:")
    print("-" * 80)
    print(etapa.get("output", ""))


ETAPA 1

CÓDIGO EXECUTADO:
--------------------------------------------------------------------------------
# Exploratory search in the provided contexto
print("Tamanho do texto:", len(contexto))

# Keywords to search
keywords = ["Capitu", "Bentinho", "Bento", "Escobar", "Ezequiel", "olhos de ressaca", "ciúme", "suspeita", "desconfiança", "Capitu e Escobar"]
results = {}

for kw in keywords:
    results[kw] = contexto.count(kw)

results

SAÍDA DO CÓDIGO:
--------------------------------------------------------------------------------
{'Capitu': 1, 'Bentinho': 57, 'Bento': 1, 'Escobar': 112, 'Ezequiel': 51, 'olhos de ressaca': 5, 'ciúme': 0, 'suspeita': 12, 'desconfiança': 2, 'Capitu e Escobar': 0}

ETAPA 2

CÓDIGO EXECUTADO:
--------------------------------------------------------------------------------
# Mostrar múltiplas ocorrências de palavras-chave com contexto para inspeção
import re

terms = ["Capitu", "Escobar", "Bentinho", "Ezequiel", "olhos de ressaca", "suspeita", "semelhan